# JSON-Analyse: Pointing Game, Mean Mass Ratio und Mean IoU

Dieses Notebook berechnet die Metriken je Methode (`gradcam`, `rollout`, `drise`) und gruppiert sie:

- nach Klassenpräfix im Dateinamen (`A`, `B`, `E`, `G`)
- nach Bit-Tiefe direkt vor der Endung (`8bit`, `16bit`)

Beispiel-Dateiname:

`A0007_1.3.6.1.4.1.14519.5.2.1.6655.2359.118972525676913596910705207496_16bit.png`

## Erwartete JSON-Struktur

Das Notebook geht davon aus, dass die JSON die Bereiche enthält:

- `pointing_game`
- `mass_in_bounding_box`
- `iou`

mit jeweils Untereinträgen pro Methode, z. B. `gradcam`, `rollout`, `drise`.

Falls `mass_in_bounding_box` oder `iou` in deiner Datei anders heißen oder anders aufgebaut sind, musst du unten nur die entsprechenden Keys anpassen.


In [2]:
import json
import re
from collections import defaultdict
from statistics import mean

import pandas as pd


## 1) JSON-Pfad setzen

Trage hier den Pfad zu deiner JSON-Datei ein.

In [3]:
json_path = 'results_eval (1).json'  # <- hier anpassen\n

## 2) Hilfsfunktionen

In [4]:
CLASS_PATTERN = re.compile(r'^([ABEG])')
BIT_PATTERN = re.compile(r'_(8bit|16bit)\.[^.]+$', re.IGNORECASE)


def parse_filename(file_name: str):
    """
    Extrahiert:
      - Klassenpräfix am Anfang: A/B/E/G
      - Bit-Tiefe direkt vor der Endung: 8bit/16bit
    """
    class_match = CLASS_PATTERN.match(file_name)
    bit_match = BIT_PATTERN.search(file_name)

    cls = class_match.group(1) if class_match else 'UNKNOWN'
    bit = bit_match.group(1).lower() if bit_match else 'unknown'
    return cls, bit


def safe_mean(values):
    vals = [v for v in values if v is not None]
    return mean(vals) if vals else None


def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def build_index_by_filename(entries, value_key):
    """
    Baut ein Mapping:
      file_name -> value

    Falls ein Dateiname mehrfach vorkommt, wird der Mittelwert dieser Werte genommen.

    Sonderfall:
    - Für pointing_game wird `hit` (True/False) auf 1.0/0.0 gemappt.
    - Nur `valid=True` wird berücksichtigt.
    """
    tmp = defaultdict(list)

    for item in entries:
        file_name = item.get('file_name')
        if not file_name:
            continue

        value = item.get(value_key)

        if value_key == 'hit':
            if not item.get('valid', False):
                continue
            if value is not None:
                tmp[file_name].append(1.0 if value else 0.0)
        else:
            if isinstance(value, (int, float)):
                tmp[file_name].append(float(value))

    return {k: safe_mean(v) for k, v in tmp.items() if v}


def aggregate_group_scores(file_to_value):
    by_class = defaultdict(list)
    by_bit = defaultdict(list)

    for file_name, value in file_to_value.items():
        cls, bit = parse_filename(file_name)
        by_class[cls].append(value)
        by_bit[bit].append(value)

    by_class = {k: safe_mean(v) for k, v in sorted(by_class.items())}
    by_bit = {k: safe_mean(v) for k, v in sorted(by_bit.items())}
    return by_class, by_bit


def flatten_results(metric_name, results_dict):
    rows = []
    for method, groups in results_dict.items():
        for group, value in groups['by_class'].items():
            rows.append({
                'metric': metric_name,
                'method': method,
                'group_type': 'class',
                'group': group,
                'value': value,
            })
        for group, value in groups['by_bit'].items():
            rows.append({
                'metric': metric_name,
                'method': method,
                'group_type': 'bit_depth',
                'group': group,
                'value': value,
            })
    return rows


## 3) Analyse ausführen

In [5]:
data = load_json(json_path)

# Erwartete Struktur:
# data['pointing_game']['gradcam'] = [...]
# data['mass_in_bounding_box']['gradcam'] = [...]
# data['iou']['gradcam'] = [...]
#
# Robuste Variante:
# Falls die Datei stattdessen 'mass_in_box' verwendet, wird das ebenfalls akzeptiert.

pointing_game = data.get('pointing_game', {})
mass_in_box = data.get('mass_in_bounding_box', data.get('mass_in_box', {}))
iou = data.get('iou', {})

methods = sorted(set(pointing_game.keys()) | set(mass_in_box.keys()) | set(iou.keys()))

pointing_results = {}
mass_results = {}
iou_results = {}

for method in methods:
    # Pointing Game
    pg_entries = pointing_game.get(method, [])
    pg_index = build_index_by_filename(pg_entries, 'hit')
    pg_by_class, pg_by_bit = aggregate_group_scores(pg_index)
    pointing_results[method] = {
        'by_class': pg_by_class,
        'by_bit': pg_by_bit,
    }

    # Mean Mass Ratio
    mib_entries = mass_in_box.get(method, [])
    mib_index = build_index_by_filename(mib_entries, 'mass_ratio')
    mib_by_class, mib_by_bit = aggregate_group_scores(mib_index)
    mass_results[method] = {
        'by_class': mib_by_class,
        'by_bit': mib_by_bit,
    }

    # Mean IoU
    iou_entries = iou.get(method, [])
    iou_index = build_index_by_filename(iou_entries, 'iou')
    iou_by_class, iou_by_bit = aggregate_group_scores(iou_index)
    iou_results[method] = {
        'by_class': iou_by_class,
        'by_bit': iou_by_bit,
    }

all_rows = []
all_rows.extend(flatten_results('pointing_game_score', pointing_results))
all_rows.extend(flatten_results('mean_mass_ratio', mass_results))
all_rows.extend(flatten_results('mean_iou', iou_results))

df = pd.DataFrame(all_rows)
df


,metric,method,group_type,group,value
0,pointing_game_score,drise,class,A,0.891513
1,pointing_game_score,drise,class,B,0.825796
2,pointing_game_score,drise,class,E,0.873418
3,pointing_game_score,drise,class,G,0.922013
4,pointing_game_score,drise,bit_depth,16bit,0.882128
5,pointing_game_score,drise,bit_depth,8bit,0.898484
6,pointing_game_score,gradcam,class,A,0.203285
7,pointing_game_score,gradcam,class,B,0.351759
8,pointing_game_score,gradcam,class,E,0.721519
9,pointing_game_score,gradcam,class,G,0.098113


## 4) Ergebnisse übersichtlich anzeigen

In [ ]:
for metric in df['metric'].unique():
    print('\n' + '=' * 80)
    print(metric)
    print('=' * 80)
    display(df[df['metric'] == metric].sort_values(['method', 'group_type', 'group']).reset_index(drop=True))


## 5) Optional als CSV speichern

In [ ]:
output_csv = 'grouped_metrics.csv'
df.to_csv(output_csv, index=False)
print(f'CSV gespeichert: {output_csv}')


## Hinweis

Wenn deine JSON **nur** `summary` und `pointing_game` enthält, aber keine Rohdatenblöcke für `mass_in_bounding_box` und `iou`, dann können `mean_mass_ratio` und `mean_iou` nicht nach Klasse oder Bit-Tiefe neu berechnet werden.

In diesem Fall brauchst du die Einzelwerte pro Bild in etwa dieser Form:

```json
{
  "mass_in_bounding_box": {
    "gradcam": [
      {
        "file_name": "A0007_..._16bit.png",
        "mass_ratio": 0.123
      }
    ]
  },
  "iou": {
    "gradcam": [
      {
        "file_name": "A0007_..._16bit.png",
        "iou": 0.456
      }
    ]
  }
}
```
